# Assignment 09 - GRAPH

이 노트북은 다음 내용을 포함합니다.

1. 그래프 생성
2. DFS 애니메이션
3. BFS 애니메이션
4. Prim 알고리즘 애니메이션
5. Kruskal 알고리즘 애니메이션


## 공통 설정

In [ ]:

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.font_manager as fm
import networkx as nx
from collections import deque
import heapq
import os

font_path = "C:/Windows/Fonts/malgun.ttf"

if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    korean_font = fm.FontProperties(fname=font_path)
else:
    font_path = "C:/Windows/Fonts/gulim.ttc"
    fm.fontManager.addfont(font_path)
    korean_font = fm.FontProperties(fname=font_path)

plt.rcParams["font.family"] = korean_font.get_name()
plt.rcParams["axes.unicode_minus"] = False

NODE_LABELS = {
    0: '소테\\n(성북구)',
    1: '유림닭도리탕\\n(강서구)',
    2: '오몬자\\n(마포구)',
    3: '남산광어\\n(용산구)',
    4: '타코스퀘어\\n(성동구)',
    5: '프레고클럽\\n(성동구)',
    6: '푸주옥\\n(송파구)',
}

EDGES = [
    (0,1,29),(0,2,27),(0,3,25),(0,4,25),(0,5,25),
    (1,2,20),(1,3,26),(1,6,22),
    (2,3,29),(2,5,39),
    (3,5,21),(3,6,30),
    (4,5,21),(4,6,30),
    (5,6,27),
]

POS = {
    0:(3.4,4.2), 1:(0.8,2.3), 2:(1.8,2.5),
    3:(2.8,1.5), 4:(4.6,3.8), 5:(4.9,2.5), 6:(5.6,1.2),
}

G = nx.Graph()

for nid in NODE_LABELS:
    G.add_node(nid)

for u, v, w in EDGES:
    G.add_edge(u, v, weight=w)

EDGE_LABEL = {(u,v): w for u,v,w in EDGES}

C_DEFAULT  = '#EEEDFE'
C_ACTIVE   = '#EF9F27'
C_DONE     = '#534AB7'
C_REJECT   = '#E24B4A'
C_EDGE_DEF = '#B4B2A9'
C_EDGE_ACT = '#534AB7'

def draw_graph(ax, node_colors, edge_colors, edge_widths, title):
    ax.clear()
    ax.set_facecolor('#FAFAF8')
    ax.axis('off')

    nx.draw_networkx_edges(
        G, POS, ax=ax,
        edge_color=edge_colors,
        width=edge_widths,
        alpha=0.85
    )

    nx.draw_networkx_nodes(
        G, POS, ax=ax,
        node_color=node_colors,
        node_size=2000,
        edgecolors=C_EDGE_ACT,
        linewidths=1.2
    )

    for node, (x, y) in POS.items():
        ax.text(
            x, y,
            NODE_LABELS[node],
            fontsize=9,
            color='white',
            ha='center',
            va='center',
            fontproperties=korean_font
        )

    for (u, v), w in EDGE_LABEL.items():
        x1, y1 = POS[u]
        x2, y2 = POS[v]

        mx = (x1 + x2) / 2
        my = (y1 + y2) / 2

        ax.text(
            mx, my,
            str(w),
            fontsize=9,
            color='#444',
            ha='center',
            va='center',
            fontproperties=korean_font,
            bbox=dict(
                boxstyle='round,pad=0.2',
                fc='white',
                alpha=0.8,
                ec='none'
            )
        )

    ax.set_title(
        title,
        fontsize=11,
        pad=12,
        color='#2C2C2A',
        fontproperties=korean_font
    )

def edge_match(u, v, target):
    return target and ((u,v)==target or (v,u)==target)

def edge_in_list(u, v, lst):
    return any((u,v)==e or (v,u)==e for e in lst)


## DFS 애니메이션

In [ ]:

def dfs_steps(start=0):
    steps = []
    visited = set()
    stack = [(start, None)]

    while stack:
        node, edge = stack.pop()

        if node in visited:
            continue

        visited.add(node)
        steps.append((node, edge))

        for nb in sorted(G.neighbors(node), reverse=True):
            if nb not in visited:
                stack.append((nb, (node, nb)))

    return steps

dfs = dfs_steps(0)

fig, ax = plt.subplots(figsize=(10, 7))

def dfs_frame(i):
    visited_set = {dfs[j][0] for j in range(i + 1)}
    cur, cur_e = dfs[i]

    used_edges = [
        dfs[j][1]
        for j in range(1, i + 1)
        if dfs[j][1]
    ]

    node_colors = [
        C_ACTIVE if n == cur
        else C_DONE if n in visited_set
        else C_DEFAULT
        for n in G.nodes()
    ]

    edge_colors = []
    edge_widths = []

    for u, v in G.edges():
        if edge_match(u, v, cur_e):
            edge_colors.append(C_ACTIVE)
            edge_widths.append(3.0)
        elif edge_in_list(u, v, used_edges):
            edge_colors.append(C_EDGE_ACT)
            edge_widths.append(2.2)
        else:
            edge_colors.append(C_EDGE_DEF)
            edge_widths.append(0.8)

    order = ', '.join(
        NODE_LABELS[dfs[j][0]].split('\\n')[0]
        for j in range(i + 1)
    )

    draw_graph(
        ax,
        node_colors,
        edge_colors,
        edge_widths,
        f'DFS | 방문 순서: {order}'
    )

ani = animation.FuncAnimation(
    fig,
    dfs_frame,
    frames=len(dfs),
    interval=700,
    repeat=False
)

plt.show()


## BFS 애니메이션

In [ ]:

def bfs_steps(start=0):
    steps = []
    visited = {start}
    queue = deque([(start, None)])

    while queue:
        node, edge = queue.popleft()
        steps.append((node, edge))

        for nb in sorted(G.neighbors(node)):
            if nb not in visited:
                visited.add(nb)
                queue.append((nb, (node, nb)))

    return steps

bfs = bfs_steps(0)

fig, ax = plt.subplots(figsize=(10, 7))

def bfs_frame(i):
    visited_set = {bfs[j][0] for j in range(i + 1)}
    cur, cur_e = bfs[i]

    used_edges = [
        bfs[j][1]
        for j in range(1, i + 1)
        if bfs[j][1]
    ]

    node_colors = [
        C_ACTIVE if n == cur
        else C_DONE if n in visited_set
        else C_DEFAULT
        for n in G.nodes()
    ]

    edge_colors = []
    edge_widths = []

    for u, v in G.edges():
        if edge_match(u, v, cur_e):
            edge_colors.append(C_ACTIVE)
            edge_widths.append(3.0)
        elif edge_in_list(u, v, used_edges):
            edge_colors.append(C_EDGE_ACT)
            edge_widths.append(2.2)
        else:
            edge_colors.append(C_EDGE_DEF)
            edge_widths.append(0.8)

    order = ', '.join(
        NODE_LABELS[bfs[j][0]].split('\\n')[0]
        for j in range(i + 1)
    )

    draw_graph(
        ax,
        node_colors,
        edge_colors,
        edge_widths,
        f'BFS | 방문 순서: {order}'
    )

ani = animation.FuncAnimation(
    fig,
    bfs_frame,
    frames=len(bfs),
    interval=700,
    repeat=False
)

plt.show()


## Prim 알고리즘 애니메이션

In [ ]:

def prim_steps(start=0):
    steps = [(start, None)]
    visited = {start}

    heap = [
        (G[start][nb]['weight'], start, nb)
        for nb in G.neighbors(start)
    ]

    heapq.heapify(heap)

    while heap and len(visited) < G.number_of_nodes():
        w, u, v = heapq.heappop(heap)

        if v in visited:
            continue

        visited.add(v)
        steps.append((v, (u, v)))

        for nb in G.neighbors(v):
            if nb not in visited:
                heapq.heappush(
                    heap,
                    (G[v][nb]['weight'], v, nb)
                )

    return steps

prim = prim_steps(0)

fig, ax = plt.subplots(figsize=(10, 7))

def prim_frame(i):
    mst_nodes = {
        prim[j][0]
        for j in range(i + 1)
    }

    mst_edges = [
        prim[j][1]
        for j in range(1, i + 1)
        if prim[j][1]
    ]

    cur, cur_e = prim[i]

    node_colors = [
        C_ACTIVE if n == cur
        else C_DONE if n in mst_nodes
        else C_DEFAULT
        for n in G.nodes()
    ]

    edge_colors = []
    edge_widths = []

    for u, v in G.edges():
        if edge_match(u, v, cur_e):
            edge_colors.append(C_ACTIVE)
            edge_widths.append(3.0)
        elif edge_in_list(u, v, mst_edges):
            edge_colors.append(C_EDGE_ACT)
            edge_widths.append(2.2)
        else:
            edge_colors.append(C_EDGE_DEF)
            edge_widths.append(0.8)

    total = sum(
        G[u][v]['weight']
        for _, (u, v) in prim[1:i + 1]
    )

    draw_graph(
        ax,
        node_colors,
        edge_colors,
        edge_widths,
        f'프림 알고리즘 | MST 누적: {total}분'
    )

ani = animation.FuncAnimation(
    fig,
    prim_frame,
    frames=len(prim),
    interval=700,
    repeat=False
)

plt.show()


## Kruskal 알고리즘 애니메이션

In [ ]:

class UF:
    def __init__(self, n):
        self.p = list(range(n))
        self.r = [0] * n

    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]

    def union(self, x, y):
        px, py = self.find(x), self.find(y)

        if px == py:
            return False

        if self.r[px] < self.r[py]:
            px, py = py, px

        self.p[py] = px

        if self.r[px] == self.r[py]:
            self.r[px] += 1

        return True

def kruskal_steps():
    uf = UF(G.number_of_nodes())
    steps = []
    mst = []

    for u, v, d in sorted(
        G.edges(data=True),
        key=lambda x: x[2]['weight']
    ):
        accepted = uf.union(u, v)

        if accepted:
            mst.append((u, v))

        steps.append(
            ((u, v), accepted, list(mst))
        )

    return steps

kruskal = kruskal_steps()

fig, ax = plt.subplots(figsize=(10, 7))

def kruskal_frame(i):
    (cu, cv), accepted, mst_edges = kruskal[i]

    rejected_edges = [
        (u, v)
        for (u, v), acc, _ in kruskal[:i]
        if not acc
    ]

    mst_nodes = {
        n
        for u, v in mst_edges
        for n in (u, v)
    }

    node_colors = []

    for n in G.nodes():
        if n in (cu, cv):
            if accepted:
                node_colors.append(C_ACTIVE)
            else:
                node_colors.append(C_REJECT)
        elif n in mst_nodes:
            node_colors.append(C_DONE)
        else:
            node_colors.append(C_DEFAULT)

    edge_colors = []
    edge_widths = []

    current_edge = (cu, cv)

    for u, v in G.edges():
        if edge_match(u, v, current_edge):
            edge_colors.append(C_ACTIVE if accepted else C_REJECT)
            edge_widths.append(3.0)

        elif edge_in_list(u, v, mst_edges):
            edge_colors.append(C_EDGE_ACT)
            edge_widths.append(2.2)

        elif edge_in_list(u, v, rejected_edges):
            edge_colors.append(C_REJECT)
            edge_widths.append(0.8)

        else:
            edge_colors.append(C_EDGE_DEF)
            edge_widths.append(0.8)

    weight = G[cu][cv]['weight']

    status = '채택' if accepted else '기각'

    total = sum(
        G[u][v]['weight']
        for u, v in mst_edges
    )

    a = NODE_LABELS[cu].split('\\n')[0]
    b = NODE_LABELS[cv].split('\\n')[0]

    draw_graph(
        ax,
        node_colors,
        edge_colors,
        edge_widths,
        f'크루스칼 | {a} ↔ {b} ({weight}분) {status} | 누적: {total}분'
    )

ani = animation.FuncAnimation(
    fig,
    kruskal_frame,
    frames=len(kruskal),
    interval=700,
    repeat=False
)

plt.show()
